The purpose of this file is to convert data from original data to istances of PDPCR

In [7]:
import numpy as np 
import pandas as pd
np.random.seed(1)

### Data based on real world wr2


In [2]:
data_name = 'nyc-n100-1'
data_path_wr = f'temp/orig_data/wr2ct4r22f-2/n100/{data_name}.txt'
# data_path_wr = 'orig_data/wr2ct4r22f-2/n5000/nyc-n5000-2.txt'

In [3]:
def raw_to_df(data_path):
    """Get data from original source, convert it to a dataframe, and get related data

    Parameters
    ----------
    data_path : string
        path to data

    Returns
    -------
    _type_
        _description_
    """
    skip_rows = 11
    with open(data_path,'r',encoding='utf8') as f:
        first_10_lines = [f.readline().strip() for _ in range(10)]
        first_10_dict = {}
        # Parse the key-value pairs and store them in the dictionary
        for line in first_10_lines:
            key, value = line.split(': ', 1)
            first_10_dict[key] = value
        num_tasks = int(first_10_dict['SIZE']) # Number of first rows to skip while reading file
        for _ in range(1 + num_tasks + 1):
            next(f)
        lines = [list(map(int, line.strip().split())) for line in f if line != "EOF"]

    distance_matrix = np.array(lines)
    header_names = ["id", "x", "y", "dem", "etw", "ltw", "sd", "p", "d"]
    orig_data = pd.read_csv(data_path, sep =" ", skiprows= skip_rows, nrows=num_tasks, names=header_names)

    depot_df = orig_data[orig_data['id'] == 0][['id', 'x', 'y']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})
    depot_df['x_dest'] = depot_df['x_pickup']
    depot_df['y_dest'] = depot_df['y_pickup']
    pickups_df = orig_data[orig_data['d'] > 0][['id', 'x', 'y', 'd']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})

    # Create a DataFrame for deliveries
    deliveries_df = orig_data[orig_data['p'] > 0][['id', 'x', 'y', 'p']].rename(columns={'id': 'id_d', 'x': 'x_dest', 'y': 'y_dest'})

    # Merge pickups and deliveries based on the pickup and delivery pair columns
    result_df = pd.merge(pickups_df, deliveries_df, left_on='d', right_on='id_d')
    result_df = pd.concat([depot_df,result_df]).reset_index()
    # Drop the redundant columns
    result_df = result_df[['index', 'x_pickup', 'y_pickup', 'x_dest', 'y_dest']]

    return  result_df, num_tasks, distance_matrix

In [10]:
def gen_coop_robots(num_tasks: int, percent_sync: float = 0.1, max_robot: int = 3, probs = [0.7,0.3]) -> list[int]:
    """Generate number of robots required for each tasks

    Parameters
    ----------
    num_tasks : int
        total tasks, excluding depot
    percent_sync : float, optional
        percent of task requires more than 1 robot , by default 1
    max_robot : int, optional
        maximum of robot that each tasks may required, by default 10
    probs : list
        probs = [0.7,0.3] means 70% coop task will need 2 robots, 30% coop tasks need 3 robots

    Returns
    -------
    list[int]
        _description_
    """

    # prob = [prob_2,prob_3] + [(1 -prob_2 - prob_3- prob_4)/(max_robot-4) for _ in range(max_robot-4) ]
    # prob = [0.4,]
    random_values = np.random.rand(num_tasks)
    coop_robots = np.where(random_values > percent_sync, 1, np.random.choice(np.arange(2, max_robot + 1), size = num_tasks, p=probs))

    return coop_robots.tolist()

In [11]:
# Add process time
def get_process_times(num_tasks, distance_matrix):
    num_pd_tasks = int((num_tasks-1)/2)
    process_times = [0]
    for i in range(1,num_pd_tasks+1):
        process_times.append(distance_matrix[i][i+num_pd_tasks])
    
    return process_times

# result_df['process_time'] = process_times

In [12]:
first_names =  ['bar']
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']

In [9]:
first_names =  ['bar']
sec_names = ['100','200','400','600','800','1000']
third_names = ['1','2','3','4','5']
percent_sync = 0.1
max_robots = 3
for i in first_names:
    for j in sec_names:
        for k in third_names:
            data_name =f'bar-n{j}-{k}'
            data_path = f'temp/orig_data/wr2ct4r22f-2/n{j}/{data_name}.txt'
            
            result_df, num_tasks, distance_matrix = raw_to_df(data_path)
            robots_required = gen_coop_robots(int((num_tasks-1)/2), percent_sync, max_robots)
            robots_required = [0] + robots_required #for depot

            process_time = get_process_times(num_tasks, distance_matrix)

            result_df['robots_required'] = robots_required
            result_df['process_time'] = process_time

            result_df.to_csv(f'./temp/converted/{data_name}-{max_robots}.csv',index=False)
            np.savetxt(f'./temp/converted/{data_name}-dist.csv',distance_matrix,fmt ='%s')
            print(f'Done {data_name}')

Done bar-n100-1
Done bar-n100-2
Done bar-n100-3
Done bar-n100-4
Done bar-n100-5
Done bar-n200-1
Done bar-n200-2
Done bar-n200-3
Done bar-n200-4
Done bar-n200-5
Done bar-n400-1
Done bar-n400-2
Done bar-n400-3
Done bar-n400-4
Done bar-n400-5
Done bar-n600-1
Done bar-n600-2
Done bar-n600-3
Done bar-n600-4
Done bar-n600-5
Done bar-n800-1
Done bar-n800-2
Done bar-n800-3
Done bar-n800-4
Done bar-n800-5
Done bar-n1000-1
Done bar-n1000-2
Done bar-n1000-3
Done bar-n1000-4
Done bar-n1000-5


In [6]:
data_name = 'nyc-n100-1'
data_path_wr = f'temp/orig_data/wr2ct4r22f-2/n100/{data_name}.txt'
# data_path_wr = 'orig_data/wr2ct4r22f-2/n5000/nyc-n5000-2.txt'

In [7]:
df_result, num_tasks, dist_matrix = raw_to_df(data_path_wr,)

### Li and lim data

In [5]:
data_name = 'lc101'
data_path_li = f'temp/orig_data/li/pdp_100/{data_name}.txt'
data_path = data_path_li
# data_path_wr = 'orig_data/wr2ct4r22f-2/n5000/nyc-n5000-2.txt'

In [2]:
def li_raw_to_df(data_path):  
    with open(data_path,'r',encoding='utf8') as f:
        first_line = f.readline().strip().split()
        num_robots = int(first_line[0])

    header_names = ["id", "x", "y", "dem", "etw", "ltw", "sd", "p", "d"]
    orig_data = pd.read_csv(data_path, sep ="\t", skiprows= 1,names=header_names)

    depot_df = orig_data[orig_data['id'] == 0][['id', 'x', 'y']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})
    depot_df['x_dest'] = depot_df['x_pickup']
    depot_df['y_dest'] = depot_df['y_pickup']
    pickups_df = orig_data[orig_data['d'] > 0][['id', 'x', 'y', 'd']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})

    deliveries_df = orig_data[orig_data['p'] > 0][['id', 'x', 'y', 'p']].rename(columns={'id': 'id_d', 'x': 'x_dest', 'y': 'y_dest'})

    result_df = pd.merge(pickups_df, deliveries_df, left_on='d', right_on='id_d')
    result_df = pd.concat([depot_df,result_df]).reset_index()
    result_df = result_df[['x_pickup', 'y_pickup', 'x_dest', 'y_dest']]
    # round down to 2 decimal

    result_df['process_time'] = np.sqrt((result_df['x_pickup'] - result_df['x_dest'])**2 + (result_df['y_pickup'] - result_df['y_dest'])**2).round(2)
    result_df.reset_index(inplace = True)
    
    return result_df, num_robots


In [3]:
def _gen_coop_robots_uniform(num_tasks: int, max_robot: int, percent_sync: float = 1) -> list[int]:
    """Generate number of robots required for each tasks

    Parameters
    ----------
    num_tasks : int
        total tasks, excluding depot
    percent_sync : float, optional
        percent of task requires more than 1 robot , by default 1
    max_robot : int, optional
        maximum of robot that each tasks may required, by default 10
    probs : list
        probs = [0.7,0.3] means 70% coop task will need 2 robots, 30% coop tasks need 3 robots

    Returns
    -------
    list[int]
        _description_
    """

    #Return num_task value from 1 to max_robot according to uniform distribution
    coop_robots = np.random.randint(1,max_robot+1,num_tasks)

    return coop_robots.tolist()

In [23]:
df_result,num_robots = li_raw_to_df(data_path_li)
robots_required = _gen_coop_robots_uniform(len(df_result)-1, 5)
robots_required = [0] + robots_required
df_result['robots_required'] = robots_required


In [44]:
df_result.to_csv(f'./temp/converted//lim/{data_name}-{5}-uniform.csv',index=False)

In [4]:

first_names = ['LC']
sec_names = [i for i in range(1,11)]
third_names = [2,4,6,8,10]

In [12]:
max_robots = 5
for i in first_names:
    for j in sec_names:
        for k in third_names:
            try:
                data_name =f'{i}1_{k}_{j}'
                data_path = f'temp/orig_data/li/pdp_{k}00/{data_name}.txt'
                df_result,num_robots = li_raw_to_df(data_path)
                robots_required = _gen_coop_robots_uniform(len(df_result)-1, max_robots)
                robots_required = [0] + robots_required
                df_result['robots_required'] = robots_required
                df_result.to_csv(f'./data/lim/{k}00/{data_name}-{max_robots}-uniform.csv',index=False)
            except Exception as e:
                print(e)
                print(f'Error {data_name}')

NameError: name 'first_names' is not defined

In [ ]:
with open(data_path,'r',encoding='utf8') as f:
    first_line = f.readline().strip().split()
    num_robots = int(first_line[0])

header_names = ["id", "x", "y", "dem", "etw", "ltw", "sd", "p", "d"]
orig_data = pd.read_csv(data_path, sep ="\t", skiprows= 1,names=header_names)

depot_df = orig_data[orig_data['id'] == 0][['id', 'x', 'y']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})
depot_df['x_dest'] = depot_df['x_pickup']
depot_df['y_dest'] = depot_df['y_pickup']
pickups_df = orig_data[orig_data['d'] > 0][['id', 'x', 'y', 'd']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})
deliveries_df = orig_data[orig_data['p'] > 0][['id', 'x', 'y', 'p']].rename(columns={'id': 'id_d', 'x': 'x_dest', 'y': 'y_dest'})

result_df = pd.merge(pickups_df, deliveries_df, left_on='d', right_on='id_d')
result_df = pd.concat([depot_df,result_df]).reset_index()
result_df = result_df[['x_pickup', 'y_pickup', 'x_dest', 'y_dest']]
# round down to 2 decimal

result_df['process_time'] = np.sqrt((result_df['x_pickup'] - result_df['x_dest'])**2 + (result_df['y_pickup'] - result_df['y_dest'])**2).round(2)
result_df.reset_index(inplace = True)
    


In [21]:
with open(data_path,'r',encoding='utf8') as f:
    first_line = f.readline().strip().split()
    num_robots = int(first_line[0])

header_names = ["id", "x", "y", "dem", "etw", "ltw", "sd", "p", "d"]
orig_data = pd.read_csv(data_path, sep ="\t", skiprows= 1,names=header_names)

depot_df = orig_data[orig_data['id'] == 0][['id', 'x', 'y']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})
depot_df['x_dest'] = depot_df['x_pickup']
depot_df['y_dest'] = depot_df['y_pickup']
pickups_df = orig_data[orig_data['d'] > 0][['id', 'x', 'y', 'd']].rename(columns={'id': 'index', 'x': 'x_pickup', 'y': 'y_pickup'})
deliveries_df = orig_data[orig_data['p'] > 0][['id', 'x', 'y', 'p']].rename(columns={'id': 'id_d', 'x': 'x_dest', 'y': 'y_dest'})

result_df = pd.merge(pickups_df, deliveries_df, left_on='d', right_on='id_d')
result_df = pd.concat([depot_df,result_df]).reset_index()

In [22]:
result_df

,level_0,index,x_pickup,y_pickup,x_dest,y_dest,d,id_d,p
0,0,0,40,50,40,50,NaN,NaN,NaN
1,0,3,42,66,45,65,75.0,75.0,3.0
2,1,5,42,65,40,66,7.0,7.0,5.0
3,2,6,40,69,45,70,2.0,2.0,6.0
4,3,8,38,68,35,66,10.0,10.0,8.0
5,4,9,38,70,42,68,4.0,4.0,9.0
6,5,11,35,69,45,68,1.0,1.0,11.0
7,6,13,22,75,18,75,17.0,17.0,13.0
8,7,16,20,85,22,85,14.0,14.0,16.0
9,8,18,15,75,25,85,12.0,12.0,18.0
